Finding 1: Feature Importance & Target Leakage

The Claim:
 The exploratory Random Forest model identifies "Average Position" and "Impressions" as the top predictive features for a page's "Health Score". 
  
  Methodology Question:
 Does this model suffer from direct target leakage? Because Health Score is explicitly calculated using Impressions (30 pts) and Position (30 pts), the model appears to be reverse-engineering its own grading rubric rather than discovering independent predictive signals.

Finding 2: Classification Accuracy & Client Leakage

The Claim: 
The Logistic Regression model achieved 71% holdout accuracy when predicting whether a page would grow or decline using an 80/20 train/test split across 57 brands. 

 Methodology Question:
 Does a random 80/20 split across the entire portfolio cause group leakage by placing pages from the same brand in both the training and test sets? If the model is memorizing the specific patterns of these 57 brands, this 71% accuracy may not hold up when evaluating a completely new 58th brand. 

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Load Data & Target exactly as defined in Week 5
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
PAGE_1_THRESHOLD = 10
BAD_CTR_THRESHOLD = 0.01 
rule_mask = (df['avg_position'] <= PAGE_1_THRESHOLD) & (df['ctr'] < BAD_CTR_THRESHOLD)
df['action_label'] = 'None'
df.loc[rule_mask, 'action_label'] = 'CTR-fix'

# Features (X) and Target (y)
X = df[['avg_position', 'ctr', 'impressions_90d']]
y = (df['action_label'] == 'CTR-fix').astype(int)

# --- BEFORE: The Naive Random Split ---
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_naive = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_naive.fit(X_train_naive, y_train_naive)
acc_naive = accuracy_score(y_test_naive, rf_naive.predict(X_test_naive))

# --- AFTER: The Honest Split (Grouped by Client) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))

X_train_honest, X_test_honest = X.iloc[train_idx], X.iloc[test_idx]
y_train_honest, y_test_honest = y.iloc[train_idx], y.iloc[test_idx]

rf_honest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_honest.fit(X_train_honest, y_train_honest)
acc_honest = accuracy_score(y_test_honest, rf_honest.predict(X_test_honest))

# --- COMPARISON ---
print(f"Naive Split Accuracy:  {acc_naive:.4f}")
print(f"Honest Split Accuracy: {acc_honest:.4f}")

Naive Split Accuracy:  1.0000
Honest Split Accuracy: 1.0000


I transitioned my Week 5 Random Forest model from a standard random split to a grouped split mapped to client_hash_id. The original random split allowed data from the same clients to appear in both training and validation sets. By enforcing a strict group split, the model is now evaluated on completely unseen clients, providing an honest baseline metric for real-world application.

3. Leakage Audit  
The honest split did not reduce the accuracy because the model suffers from massive target leakage. The target variable (action_label) was procedurally generated using a hard-coded formula (avg_position <= 10 and ctr < 0.01). Because those exact same variables were passed into the model as predictive features (X), the model achieved 1.0 accuracy by simply reverse-engineering the mathematical rule, not by learning to predict SEO behavior. To fix this leakage in future iterations, the features used to calculate the target must be removed from the training inputs.

1. The Overconfident Original Claim  
Our Random Forest model accurately predicts which content pieces require a CTR-fix with 100% accuracy, proving that low click-through rates are directly caused by poor search positioning.

2. The Rewritten, Public-Safe Claim    
Based on an evaluation of the anonymized content snapshot, the model demonstrates a strong observational correlation between specific positioning thresholds and click-through rates. Rather than functioning as an automated predictor, the output serves as a decision-support heuristic to triage pages showing measured underperformance. Because the target variable was procedurally linked to input features, this model provides a directional priority signal for content reviews rather than a guaranteed causal forecast.